In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pyarrow.feather as feather

import matplotlib.pyplot as plt
import seaborn as sns 
from pathlib import Path

from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from typing import Dict, List, Tuple


from lstm import LSTMForecaster
from dataset import TimeSeriesDataset
from forecaster import MultiProductForecaster

from model_utils.plots import *
import os
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

# Import data preprocessing utilities
from EDA.preprocessing import preprocess_features, inverse_transform_target
# Import LSTM plotting utilities
from model_utils.plots import *


# Dataset Definition

## Training Visualization Features

The `MultiProductForecaster` uses plotting utilities from the `plots.py` module for comprehensive visualization during training.

### Plots Generated for Each Product:

1. **Data Distribution** (`data_distribution.png`):
   - Time series with train/val/test splits
   - Value distribution histograms by split
   - Box plots comparing splits
   - Detailed statistics summary

2. **Loss Curves** (`loss_curves.png`):
   - Training loss evolution
   - Validation loss evolution
   - Early stopping visualization

3. **Forecast Comparison** (`forecast_comparison.png`):
   - Forecast vs actual values
   - Zoomed test period detail
   - Distribution comparison (overall vs test vs forecast)
   - Error metrics (MAE, RMSE, MAPE)

All plots are saved to: `./training_plots/store_{store_id}_item_{item_id}/`

### Plotting Module

The plotting functions are located in `plots.py` and include:
- `plot_data_distribution()` - Visualize data splits and statistics
- `plot_loss_curves()` - Show training progress
- `plot_forecast_comparison()` - Compare predictions with actual values

These functions are called automatically during training when `save_plots=True`.


In [ ]:
subset_set= True 
dataset_imputed= False 
data_andre = False

selected_store = 6269


In [ ]:
path = ''
if subset_set:
    path = '../dataset/subset_set.feather'
elif dataset_imputed:
    path = '../dataset/df_imputed.feather'
elif data_andre:
    path = '../dataset/data_andre.feather'

In [ ]:
table = feather.read_table(path, memory_map=True)
df_selected = table.to_pandas()
#df_selected.head()
df_selected.columns


# LSTM with Fixed Lookback Window

## Pipeline Overview
1. **Preprocess**: Scale numericals and apply log1p transformation to target
2. **Create sequences**: Fixed lookback window [t-lookback:t] -> [t+1]
3. **LSTM input**: (batch_size, lookback, 1)
4. **Prediction**: One-step-ahead, then recursive forecasting with sliding window

## Lookback Window Approach
- Uses last N days (configurable: 7, 14, 30, etc.) to predict next day
- Simpler than variable-length sequences - no collate function needed
- During forecasting: maintains sliding window of predictions

In [ ]:
if 'date' not in df_selected.columns:
    if df_selected.index.name == 'date' or isinstance(df_selected.index, pd.DatetimeIndex):
        df_selected = df_selected.reset_index()
        print(" Date column recovered from index")

if 'date' in df_selected.columns:
    df_selected['date'] = pd.to_datetime(df_selected['date'])
    print(f" Date column is now datetime: {df_selected['date'].dtype}")
else:
    print("⚠️ Warning: 'date' column still missing!")

In [ ]:

identifiers = ['date', 'store_id', 'item_id', 'value']
exogenous_features = [col for col in df_selected.columns if col.startswith('promo_')]

print(f"  {exogenous_features}")


In [ ]:
TRAIN_RATIO = 0.6      # 60% train (and lookback)
VAL_RATIO = 0.2        # 20% validation
TEST_RATIO = 0.2       # 20% test (forecast horizon)
LOOKBACK_DAYS = 30     # Fixed lookback window size (7, 14, 30, etc.)
BATCH_SIZE = 32
HIDDEN_SIZE = 128
NUM_LAYERS = 2
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
exogenous = False  # True to include exogenous features, False to exclude


print(f"Using device: {DEVICE}")
print(f"Lookback window: {LOOKBACK_DAYS} days")

In [ ]:
# Initialize forecaster with 60/20/20 split and fixed lookback window
forecaster = MultiProductForecaster(
    train_ratio=TRAIN_RATIO,  # 60%
    val_ratio=VAL_RATIO,      # 20%
    lookback_days=LOOKBACK_DAYS,  # Fixed lookback window
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    save_plots=True,          # Enable plot saving
    plots_base_dir='./training_plots',  # Directory for plots
    use_log1p=False,  # Apply log1p transformation to target (recommended for sales data)
    disable_preprocessing=True
)

# Train models
results = forecaster.train_all_products(
    df_selected,  # Pass RAW data - preprocessing happens inside prepare_data per-product
    store_id=selected_store,
    include_features=exogenous_features if exogenous else None  # Pass exogenous features
)

print(f"\n{'='*80}")
print(f"Training complete! Trained {len(results)} models.")
print(f"✓ Each model used:")
print(f"  - Lookback window: {LOOKBACK_DAYS} days")
print(f"  - Target: 'value' (scaled)")
print(f"  - Input shape per sample: ({LOOKBACK_DAYS}, 1)")
print(f"{'='*80}")

In [ ]:
# Check prediction value list for first product
first_product_key = list(results.keys())[0]
product_1_results = results[first_product_key]

print(f"Product: Store {first_product_key[0]}, Item {first_product_key[1]}")
print(f"Forecast horizon: {product_1_results['forecast']}")


In [ ]:


# View plots for the first trained item
if len(results) > 0:
    first_item = list(results.keys())[0]
    show_item_plots(first_item[0], first_item[1])


## Common Patterns and Solutions

Based on the diagnostics above, here are the most likely causes and solutions:

### Pattern 1: Low Variance in Scaled Data
**Symptom:** Scaled training values have std < 0.1  
**Cause:** After log1p + MinMaxScaler, data becomes too compressed  
**Solutions:**
- Try StandardScaler instead of MinMaxScaler
- Skip log1p transformation (set `use_log1p=False`)
- Use different scaling per feature

### Pattern 2: Model Learns to Predict Mean
**Symptom:** All predictions ≈ training mean, loss reduces but predictions flat  
**Cause:** MSE loss minimized by predicting the mean when model can't find patterns  
**Solutions:**
- Increase model capacity (more hidden units)
- Increase sequence length (more lookback days)
- Add regularization to force weight diversity
- Try different loss function (MAE, Huber)

### Pattern 3: Vanishing Gradients
**Symptom:** Loss doesn't decrease much, weights are very small  
**Cause:** Gradients vanish through LSTM layers  
**Solutions:**
- Reduce number of layers
- Increase learning rate
- Use gradient clipping
- Try GRU instead of LSTM

### Pattern 4: Model Not Learning
**Symptom:** Loss stays nearly constant across epochs  
**Cause:** Learning rate too low, or data has no learnable patterns  
**Solutions:**
- Increase learning rate (try 0.01 instead of 0.001)
- Verify data has temporal patterns
- Check for data leaks or preprocessing errors
- Try simpler model first (MLP) to verify data quality

### Pattern 5: Recursive Prediction Collapse
**Symptom:** First few predictions vary, then converge to constant  
**Cause:** Prediction errors compound in autoregressive forecasting  
**Solutions:**
- Train with teacher forcing probability
- Add noise during training to model
- Use attention mechanism
- Predict multiple steps ahead during training

## Quick Fixes to Try

Run one of the cells below to test potential solutions:

### QUICK FIX 3: Use StandardScaler instead of MinMaxScaler

This requires modifying the `preprocess_features` function in [EDA/preprocessing.py](../EDA/preprocessing.py#L117-L120).

**Current (MinMaxScaler):**
```python
scaler = MinMaxScaler()  # Scales to [0, 1]
```

**Change to (StandardScaler):**
```python
scaler = StandardScaler()  # Scales to mean=0, std=1
```

StandardScaler often works better for neural networks because:
- Preserves the shape of the distribution
- Doesn't compress outliers as much
- Allows negative values (more dynamic range)

## 🎯 ROOT CAUSE IDENTIFIED: Recursive Prediction Collapse

**The loguru logs show the exact problem:**

### What's Happening:
1. **Steps 1-37**: Model predicts with diverse inputs, outputs vary slightly around 0.201
2. **Step 38**: The 30-day sliding window is now filled entirely with predictions (~0.2014)
3. **Steps 38-153**: Input = [0.2014, 0.2014, ..., 0.2014] → Output = 0.2014 (forever!)

### Why This Happens:
The model learned a **fixed point** (mathematical attractor):
```
f([x, x, x, ..., x]) = x  where x ≈ 0.2014 (scaled) or 73.10 (original)
```

This is optimal for MSE loss on stationary data! If data has no strong trend, predicting the mean minimizes squared error.

### The Real Problem:
**Autoregressive forecasting with one-step training creates attractors!**

During training:
- Model learns: Given real history → Predict next value
- Learns to be conservative (predict near mean)

During inference:
- First predictions are reasonable (has real history)
- But predictions replace history in sliding window  
- After ~30 steps, history is 100% predictions
- Model sees constant input → Outputs constant value

### Why This Is Different From Training:
- **Training**: Always uses REAL historical values as input
- **Inference**: Uses PREDICTED values (which are less diverse than reality)
- This distribution shift causes collapse to the attractor